# ML-06 — Signal Audit: Do the Flags Hold?

## Lane: Refresh / Content Opportunity Scoring

This notebook audits whether candidate search signals show an observed relationship with future content movement.

The goal is not to prove ranking causes or predict Google's algorithm. The goal is to create a reliable decision-support layer for content review prioritization.

Signals are calculated only from the feature window (March 1–15, 2026).

The outcome window (March 16–31, 2026) is used only for evaluation.

All conclusions use observed, measured, and directional language.

In [29]:
%pip install -q duckdb huggingface_hub pandas numpy

In [31]:
import os
import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import whoami


HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN missing. Add token in Colab Secrets."
    )


user = whoami(token=HF_TOKEN)

print(
    "Hugging Face login successful:",
    user["name"]
)


con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")


con.execute(f"""
CREATE OR REPLACE SECRET hf
(
TYPE huggingface,
TOKEN '{safe_token}'
)
""")


REL = "hf://datasets/FlyRank/internship-warehouse"


MARCH_FACT = f"""
read_parquet(
'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""


print("DuckDB connection ready.")
print("Development month: March 2026")
print("Feature window: March 1-15")
print("Outcome window: March 16-31")

Hugging Face login successful: zafar4050
DuckDB connection ready.
Development month: March 2026
Feature window: March 1-15
Outcome window: March 16-31


## Audit Frame Definition

One row represents:

**One anonymized content page for one client.**

Feature window:

March 1–15, 2026

Outcome window:

March 16–31, 2026


The declining proxy:

A page is marked as declining when:

Outcome average daily impressions < 80% of feature-window average daily impressions.


This proxy represents observed movement only.

It does not prove:
- why movement happened;
- that a refresh will recover performance;
- any search engine ranking rule.

In [32]:
audit_frame = con.sql(f"""

SELECT

client_hash_id,
content_hash_id,


SUM(
CASE
WHEN report_date BETWEEN
DATE '2026-03-01'
AND DATE '2026-03-15'
AND gsc_data_available IS TRUE

THEN COALESCE(gsc_impressions,0)

ELSE 0
END
)

AS feature_impressions,


SUM(
CASE
WHEN report_date BETWEEN
DATE '2026-03-01'
AND DATE '2026-03-15'
AND gsc_data_available IS TRUE

THEN COALESCE(gsc_clicks,0)

ELSE 0
END
)

AS feature_clicks,


AVG(
CASE
WHEN report_date BETWEEN
DATE '2026-03-01'
AND DATE '2026-03-15'
AND gsc_data_available IS TRUE

THEN gsc_avg_position

END
)

AS feature_avg_position,


STDDEV_SAMP(
CASE
WHEN report_date BETWEEN
DATE '2026-03-01'
AND DATE '2026-03-15'
AND gsc_data_available IS TRUE

THEN gsc_avg_position

END
)

AS feature_position_volatility,


COUNT(
DISTINCT CASE
WHEN report_date BETWEEN
DATE '2026-03-01'
AND DATE '2026-03-15'

AND gsc_data_available IS TRUE

THEN report_date

END
)

AS feature_available_days,


SUM(
CASE
WHEN report_date BETWEEN
DATE '2026-03-16'
AND DATE '2026-03-31'

AND gsc_data_available IS TRUE

THEN COALESCE(gsc_impressions,0)

ELSE 0

END
)

AS outcome_impressions,


COUNT(
DISTINCT CASE
WHEN report_date BETWEEN
DATE '2026-03-16'
AND DATE '2026-03-31'

AND gsc_data_available IS TRUE

THEN report_date

END
)

AS outcome_available_days


FROM {MARCH_FACT}


GROUP BY

client_hash_id,
content_hash_id


""").df()


print(
    "Raw rows:",
    len(audit_frame)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw rows: 331437


In [33]:
audit_frame = audit_frame[
    (audit_frame.feature_impressions >= 100)
    &
    (audit_frame.feature_available_days >= 5)
    &
    (audit_frame.outcome_available_days >= 5)

].copy()


audit_frame["feature_daily_impressions"] = (
    audit_frame.feature_impressions /
    audit_frame.feature_available_days
)


audit_frame["outcome_daily_impressions"] = (
    audit_frame.outcome_impressions /
    audit_frame.outcome_available_days
)



audit_frame["is_declining_proxy"] = (

audit_frame.outcome_daily_impressions

<

0.80 *

audit_frame.feature_daily_impressions

).astype(int)



audit_frame["feature_ctr"] = (

audit_frame.feature_clicks /

audit_frame.feature_impressions

)


audit_frame["feature_position_volatility"] = (
audit_frame.feature_position_volatility
.fillna(
audit_frame.feature_position_volatility.median()
)
)


print(
"Eligible rows:",
len(audit_frame)
)


print(
"Decline rate:",
round(
audit_frame.is_declining_proxy.mean(),
3
)
)

Eligible rows: 76201
Decline rate: 0.345


# 1. Distributions

Before choosing thresholds for the later baseline, I inspected the distributions of the main feature-window signals.

The inspected signals are:

- impressions;
- clicks;
- CTR;
- average search position;
- position volatility.

Impressions and clicks are expected to have heavy tails. This means that most pages receive modest exposure, while a small number of pages receive extremely large values.

A heavy-tailed signal should not be used directly in an uncapped linear score because a few extreme pages could dominate the ranked queue.

In [34]:
distribution_columns = [
    "feature_impressions",
    "feature_clicks",
    "feature_ctr",
    "feature_avg_position",
    "feature_position_volatility",
]

distribution_summary = (
    audit_frame[distribution_columns]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .T
)

distribution_summary

,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
feature_impressions,76201.0,1642.757077,3603.414164,100.000000,243.000000,590.000000,1570.000000,3816.000000,6410.000000,16344.000000,161575.000000
feature_clicks,76201.0,4.945408,18.371146,0.000000,0.000000,1.000000,4.000000,12.000000,21.000000,60.000000,2395.000000
feature_ctr,76201.0,0.002887,0.004640,0.000000,0.000000,0.001346,0.004172,0.007722,0.010695,0.020147,0.213592
feature_avg_position,76201.0,12.589168,12.561123,0.012229,4.366972,7.588764,16.473467,30.042364,38.392042,61.296403,127.620709
feature_position_volatility,76201.0,4.897639,4.705462,0.020824,1.501515,3.403126,6.636631,11.109750,14.330018,21.731766,130.995590


In [35]:
heavy_tail_summary = pd.DataFrame(
    {
        "signal": distribution_columns,
        "median": [
            audit_frame[column].median()
            for column in distribution_columns
        ],
        "p95": [
            audit_frame[column].quantile(0.95)
            for column in distribution_columns
        ],
        "p99": [
            audit_frame[column].quantile(0.99)
            for column in distribution_columns
        ],
        "maximum": [
            audit_frame[column].max()
            for column in distribution_columns
        ],
    }
)

heavy_tail_summary["p99_to_median_ratio"] = (
    heavy_tail_summary["p99"]
    / heavy_tail_summary["median"].replace(0, np.nan)
)

heavy_tail_summary

,signal,median,p95,p99,maximum,p99_to_median_ratio
0,feature_impressions,590.000000,6410.000000,16344.000000,161575.000000,27.701695
1,feature_clicks,1.000000,21.000000,60.000000,2395.000000,60.000000
2,feature_ctr,0.001346,0.010695,0.020147,0.213592,14.968864
3,feature_avg_position,7.588764,38.392042,61.296403,127.620709,8.077258
4,feature_position_volatility,3.403126,14.330018,21.731766,130.995590,6.385825


In [36]:
heavy_tail_summary["heavy_tail_verdict"] = np.where(
    heavy_tail_summary["p99_to_median_ratio"] >= 10,
    "HEAVY TAIL",
    "MODERATE / LIMITED TAIL",
)

heavy_tail_summary[
    [
        "signal",
        "median",
        "p99",
        "maximum",
        "p99_to_median_ratio",
        "heavy_tail_verdict",
    ]
]

,signal,median,p99,maximum,p99_to_median_ratio,heavy_tail_verdict
0,feature_impressions,590.000000,16344.000000,161575.000000,27.701695,HEAVY TAIL
1,feature_clicks,1.000000,60.000000,2395.000000,60.000000,HEAVY TAIL
2,feature_ctr,0.001346,0.020147,0.213592,14.968864,HEAVY TAIL
3,feature_avg_position,7.588764,61.296403,127.620709,8.077258,MODERATE / LIMITED TAIL
4,feature_position_volatility,3.403126,21.731766,130.995590,6.385825,MODERATE / LIMITED TAIL


### Distribution Interpretation

The median should be compared with the 95th percentile, 99th percentile, and maximum.

A large 99th-percentile-to-median ratio indicates that a small number of pages have values far above the typical page.

For the later baseline, impressions and clicks should therefore be transformed, capped, bucketed, or converted into percentile ranks instead of being used as unrestricted raw values.

This prevents a small number of extremely large pages from dominating the complete action score.

# 2. Signal Tests

I test three safe signals calculated only from March 1–15, 2026.

The outcome proxy from March 16–31 is used only to evaluate each signal.

The three tested signals are:

1. Visibility / impressions
2. Position volatility
3. Average search position

Each signal receives one verdict:

- `CONFIRMED`
- `OPPOSITE`
- `MIXED`
- `FALSE`

For this audit, an absolute decline-rate range below three percentage points is treated as too small for a strong directional verdict.

This three-percentage-point threshold is a practical audit policy, not a universal statistical law.

## Signal Test 1 — Visibility / Impressions

### Hypothesis

Pages with more impressions represent greater business exposure.

This test checks whether later decline rates differ across visibility buckets.

Visibility is not assumed to cause decline. Even if the relationship is mixed, visibility may still be useful as an impact or prioritization weight.

In [37]:
audit_frame["visibility_bucket"] = pd.cut(
    audit_frame["feature_impressions"],
    bins=[
        99,
        499,
        1_999,
        9_999,
        np.inf,
    ],
    labels=[
        "100-499",
        "500-1,999",
        "2,000-9,999",
        "10,000+",
    ],
    include_lowest=True,
)

signal_1_table = (
    audit_frame
    .groupby(
        "visibility_bucket",
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        declining_pages=("is_declining_proxy", "sum"),
        decline_rate=("is_declining_proxy", "mean"),
        median_impressions=("feature_impressions", "median"),
    )
    .reset_index()
)

signal_1_table["decline_rate_pct"] = (
    100 * signal_1_table["decline_rate"]
)

signal_1_table

,visibility_bucket,n,declining_pages,decline_rate,median_impressions,decline_rate_pct
0,100-499,34590,12062,0.348714,224.0,34.871350
1,"500-1,999",26261,8784,0.334488,922.0,33.448840
2,"2,000-9,999",13474,4670,0.346593,3443.5,34.659344
3,"10,000+",1876,748,0.398721,14786.5,39.872068


In [38]:
visibility_rates = (
    signal_1_table["decline_rate"]
    .dropna()
    .to_numpy()
)

visibility_range = (
    visibility_rates.max()
    - visibility_rates.min()
)

if visibility_range < 0.03:
    signal_1_verdict = "FALSE"

elif np.all(np.diff(visibility_rates) >= 0):
    signal_1_verdict = "CONFIRMED"

elif np.all(np.diff(visibility_rates) <= 0):
    signal_1_verdict = "OPPOSITE"

else:
    signal_1_verdict = "MIXED"

print("Signal 1 — Visibility")
print("Verdict:", signal_1_verdict)
print(
    "Decline-rate range:",
    round(float(visibility_range), 3),
)

Signal 1 — Visibility
Verdict: MIXED
Decline-rate range: 0.064


### Signal 1 Verdict

**Verdict: MIXED**

The observed decline rate did not increase consistently across all visibility buckets.

The 10,000+ impression bucket had the highest observed decline rate, while the lower and middle buckets showed an uneven pattern.

Visibility should therefore be used as a measure of potential business impact, not as stand-alone evidence that a page requires a refresh.

## Signal Test 2 — Position Volatility

### Hypothesis

Pages with unstable search positions may show a higher later decline rate.

Position volatility is calculated only from feature-window observations, so it is available before the outcome period.

A confirmed result would support using volatility as an instability signal. A mixed result would support using it only as secondary context.

In [39]:
audit_frame["volatility_bucket"] = pd.qcut(
    audit_frame[
        "feature_position_volatility"
    ].rank(method="first"),
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High",
    ],
)

signal_2_table = (
    audit_frame
    .groupby(
        "volatility_bucket",
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        declining_pages=("is_declining_proxy", "sum"),
        decline_rate=("is_declining_proxy", "mean"),
        median_volatility=(
            "feature_position_volatility",
            "median",
        ),
    )
    .reset_index()
)

signal_2_table["decline_rate_pct"] = (
    100 * signal_2_table["decline_rate"]
)

signal_2_table

,volatility_bucket,n,declining_pages,decline_rate,median_volatility,decline_rate_pct
0,Low,19051,5702,0.299302,0.939616,29.930187
1,Medium,19050,7013,0.368136,2.294394,36.813648
2,High,19050,6636,0.348346,4.836791,34.834646
3,Very High,19050,6913,0.362887,10.008863,36.288714


In [40]:
volatility_rates = (
    signal_2_table["decline_rate"]
    .dropna()
    .to_numpy()
)

volatility_range = (
    volatility_rates.max()
    - volatility_rates.min()
)

if volatility_range < 0.03:
    signal_2_verdict = "FALSE"

elif np.all(np.diff(volatility_rates) >= 0):
    signal_2_verdict = "CONFIRMED"

elif np.all(np.diff(volatility_rates) <= 0):
    signal_2_verdict = "OPPOSITE"

else:
    signal_2_verdict = "MIXED"

print("Signal 2 — Position Volatility")
print("Verdict:", signal_2_verdict)
print(
    "Decline-rate range:",
    round(float(volatility_range), 3),
)

Signal 2 — Position Volatility
Verdict: MIXED
Decline-rate range: 0.069


### Signal 2 Verdict

**Verdict: MIXED**

Low-volatility pages showed the lowest observed decline rate.

However, the decline rate did not increase consistently from medium to high to very-high volatility.

Position volatility may therefore be useful as supporting instability context, but it should not become an automatic refresh trigger.

## Signal Test 3 — Average Search Position

### Hypothesis

Pages in different search-position bands may show different later decline rates.

Average position is calculated only from the feature window.

A deeper position may indicate weaker search visibility, but it does not by itself prove that content should be refreshed.

In [41]:
audit_frame["position_bucket"] = pd.cut(
    audit_frame["feature_avg_position"],
    bins=[
        -np.inf,
        3,
        10,
        20,
        np.inf,
    ],
    labels=[
        "Top 3",
        "Page 1",
        "Page 2",
        "Deep",
    ],
)

signal_3_table = (
    audit_frame
    .groupby(
        "position_bucket",
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        declining_pages=("is_declining_proxy", "sum"),
        decline_rate=("is_declining_proxy", "mean"),
        median_position=("feature_avg_position", "median"),
    )
    .reset_index()
)

signal_3_table["decline_rate_pct"] = (
    100 * signal_3_table["decline_rate"]
)

signal_3_table

,position_bucket,n,declining_pages,decline_rate,median_position,decline_rate_pct
0,Top 3,9140,2772,0.303282,2.242445,30.328228
1,Page 1,36709,12621,0.343812,5.630492,34.381214
2,Page 2,15079,4678,0.310233,13.637607,31.023277
3,Deep,15273,6193,0.405487,30.017759,40.548681


In [42]:
position_rates = (
    signal_3_table["decline_rate"]
    .dropna()
    .to_numpy()
)

position_range = (
    position_rates.max()
    - position_rates.min()
)

if position_range < 0.03:
    signal_3_verdict = "FALSE"

elif np.all(np.diff(position_rates) >= 0):
    signal_3_verdict = "CONFIRMED"

elif np.all(np.diff(position_rates) <= 0):
    signal_3_verdict = "OPPOSITE"

else:
    signal_3_verdict = "MIXED"

print("Signal 3 — Average Search Position")
print("Verdict:", signal_3_verdict)
print(
    "Decline-rate range:",
    round(float(position_range), 3),
)

Signal 3 — Average Search Position
Verdict: MIXED
Decline-rate range: 0.102


### Signal 3 Verdict

**Verdict: MIXED**

Deep-ranking pages showed the highest observed decline rate.

However, the pattern across Top 3, Page 1, and Page 2 was not consistently increasing.

Average position is therefore useful as search context, but it should be combined with other signals rather than used as a stand-alone refresh rule.

In [43]:
signal_verdict_summary = pd.DataFrame(
    [
        {
            "signal": "Visibility / impressions",
            "verdict": signal_1_verdict,
            "practical_role": (
                "Potential impact and prioritization"
            ),
        },
        {
            "signal": "Position volatility",
            "verdict": signal_2_verdict,
            "practical_role": (
                "Supporting ranking-instability context"
            ),
        },
        {
            "signal": "Average search position",
            "verdict": signal_3_verdict,
            "practical_role": (
                "Supporting search-position context"
            ),
        },
    ]
)

signal_verdict_summary

,signal,verdict,practical_role
0,Visibility / impressions,MIXED,Potential impact and prioritization
1,Position volatility,MIXED,Supporting ranking-instability context
2,Average search position,MIXED,Supporting search-position context


# 3. The Flag-Linked Test — CTR Relative to Position

FlyRank's CTR-review logic relies on comparing CTR with search position.

Raw CTR values should not be compared across very different positions because a page in the Top 3 naturally has a different click opportunity from a page around position 18.

I therefore restricted the audit to pages with at least 500 feature-window impressions and average positions between 1 and 20. I compared every page's CTR with the median CTR of pages in the same position band.

Pages below their position-band median were marked as below-expected CTR.

This signal uses only measurements available during March 1–15. The later declining proxy is used only to evaluate the relationship.

In [46]:
print(audit_frame.columns.tolist())

['client_hash_id', 'content_hash_id', 'feature_impressions', 'feature_clicks', 'feature_avg_position', 'feature_position_volatility', 'feature_available_days', 'outcome_impressions', 'outcome_available_days', 'feature_daily_impressions', 'outcome_daily_impressions', 'is_declining_proxy', 'feature_ctr', 'visibility_bucket', 'volatility_bucket', 'position_bucket']


In [47]:
# Restrict the CTR audit to sufficiently visible pages
# in positions where CTR review is operationally meaningful.
ctr_audit = audit_frame[
    (audit_frame["feature_impressions"] >= 500)
    & (audit_frame["feature_avg_position"] > 0)
    & (audit_frame["feature_avg_position"] <= 20)
].copy()

# Position bands for fair CTR comparison
ctr_audit["ctr_position_band"] = pd.cut(
    ctr_audit["feature_avg_position"],
    bins=[0, 3, 10, 20],
    labels=["Top 3", "Page 1", "Page 2"],
    include_lowest=True,
)

# Expected CTR = median CTR inside the same position band
ctr_reference = (
    ctr_audit
    .groupby(
        "ctr_position_band",
        observed=True
    )["feature_ctr"]
    .median()
    .rename("expected_ctr_for_band")
)

ctr_audit = ctr_audit.merge(
    ctr_reference,
    left_on="ctr_position_band",
    right_index=True,
    how="left",
)

# CTR gap relative to comparable pages
ctr_audit["ctr_gap"] = (
    ctr_audit["feature_ctr"]
    - ctr_audit["expected_ctr_for_band"]
)

ctr_audit["ctr_status"] = np.where(
    ctr_audit["ctr_gap"] < 0,
    "Below band median",
    "At or above band median",
)

flag_linked_table = (
    ctr_audit
    .groupby(
        ["ctr_position_band", "ctr_status"],
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        declining_pages=("is_declining_proxy", "sum"),
        decline_rate=("is_declining_proxy", "mean"),
        median_impressions=("feature_impressions", "median"),
        median_ctr=("feature_ctr", "median"),
        median_ctr_gap=("ctr_gap", "median"),
    )
    .reset_index()
)

flag_linked_table["decline_rate_pct"] = (
    100 * flag_linked_table["decline_rate"]
)

flag_linked_table

,ctr_position_band,ctr_status,n,declining_pages,decline_rate,median_impressions,median_ctr,median_ctr_gap,decline_rate_pct
0,Top 3,At or above band median,3119,577,0.184995,1690.0,0.005789,0.002778,18.499519
1,Top 3,Below band median,3118,1158,0.371392,1550.5,0.001378,-0.001633,37.139192
2,Page 1,At or above band median,10489,2512,0.239489,1469.0,0.004399,0.002284,23.948899
3,Page 1,Below band median,10488,4389,0.418478,1310.0,0.000940,-0.001175,41.847826
4,Page 2,At or above band median,3413,657,0.192499,947.0,0.004673,0.002162,19.249927
5,Page 2,Below band median,3411,1224,0.358839,998.0,0.000830,-0.001680,35.883905


In [48]:
flag_summary = (
    ctr_audit
    .groupby(
        "ctr_status",
        observed=True,
    )
    .agg(
        n=("is_declining_proxy", "size"),
        declining_pages=("is_declining_proxy", "sum"),
        decline_rate=("is_declining_proxy", "mean"),
        median_ctr_gap=("ctr_gap", "median"),
    )
    .reset_index()
)

flag_summary["decline_rate_pct"] = (
    100 * flag_summary["decline_rate"]
)

flag_summary

,ctr_status,n,declining_pages,decline_rate,median_ctr_gap,decline_rate_pct
0,At or above band median,17021,3746,0.220081,0.002218,22.008108
1,Below band median,17017,6771,0.397896,-0.001350,39.789622


In [49]:
flag_rate_map = dict(
    zip(
        flag_summary["ctr_status"],
        flag_summary["decline_rate"],
    )
)

below_rate = flag_rate_map.get(
    "Below band median",
    np.nan,
)

above_rate = flag_rate_map.get(
    "At or above band median",
    np.nan,
)

flag_difference = below_rate - above_rate

if pd.isna(flag_difference):
    flag_verdict = "FALSE"
elif flag_difference >= 0.03:
    flag_verdict = "CONFIRMED"
elif flag_difference <= -0.03:
    flag_verdict = "OPPOSITE"
else:
    flag_verdict = "MIXED"

print("Flag-linked verdict:", flag_verdict)
print(
    "Below-minus-above decline-rate difference:",
    round(float(flag_difference), 3),
)

Flag-linked verdict: CONFIRMED
Below-minus-above decline-rate difference: 0.178


### Flag-Linked Verdict

**Verdict: CONFIRMED**

Pages with CTR below the median for their position band showed a meaningfully higher observed decline rate than pages at or above the band median.

This supports using position-adjusted low CTR as a review signal in the later baseline. However, the result remains observational and does not prove that changing metadata or content would cause recovery.

In [25]:
verdict_summary = pd.DataFrame(
    [
        {
            "signal": "Visibility / impressions",
            "verdict": signal_1_verdict,
            "role_in_baseline": (
                "Potential impact and review priority"
            ),
        },
        {
            "signal": "Position volatility",
            "verdict": signal_2_verdict,
            "role_in_baseline": (
                "Ranking-instability context"
            ),
        },
        {
            "signal": "Average position",
            "verdict": signal_3_verdict,
            "role_in_baseline": (
                "Search-position context"
            ),
        },
        {
            "signal": "CTR relative to position",
            "verdict": flag_verdict,
            "role_in_baseline": (
                "Flag-linked CTR review signal"
            ),
        },
    ]
)

verdict_summary

,signal,verdict,role_in_baseline
0,Visibility / impressions,MIXED,Potential impact and review priority
1,Position volatility,MIXED,Ranking-instability context
2,Average position,MIXED,Search-position context
3,CTR relative to position,CONFIRMED,Flag-linked CTR review signal


# 4. What This Means in Practice

The signal audit shows that no single signal should automatically decide whether a page needs a refresh.

Visibility, position volatility, and average search position produced mixed relationships with the later decline proxy. These signals may still provide useful context, but they should receive limited weight in the later baseline.

CTR relative to position produced the clearest observed signal. Pages with below-expected CTR within comparable position bands showed a higher later decline rate.

A content team should therefore use the final score as a ranked review queue. A human reviewer should confirm whether low CTR reflects weak metadata, poor intent alignment, seasonality, low-quality traffic, or another explanation before taking action.

In [50]:
final_verdict_summary = pd.DataFrame(
    [
        {
            "signal": "Visibility / impressions",
            "verdict": signal_1_verdict,
            "baseline_use": "Impact weight only",
            "reason": (
                "High visibility increases business importance, "
                "but decline rates were not consistently ordered."
            ),
        },
        {
            "signal": "Position volatility",
            "verdict": signal_2_verdict,
            "baseline_use": "Supporting context",
            "reason": (
                "Low-volatility pages had lower decline, "
                "but the remaining pattern was uneven."
            ),
        },
        {
            "signal": "Average search position",
            "verdict": signal_3_verdict,
            "baseline_use": "Context and filtering",
            "reason": (
                "Deep pages had higher decline, "
                "but the complete pattern was not monotonic."
            ),
        },
        {
            "signal": "CTR relative to position",
            "verdict": flag_verdict,
            "baseline_use": "Primary weakness signal",
            "reason": (
                "Below-position-band CTR pages showed "
                "a meaningfully higher decline rate."
            ),
        },
    ]
)

final_verdict_summary

,signal,verdict,baseline_use,reason
0,Visibility / impressions,MIXED,Impact weight only,"High visibility increases business importance,..."
1,Position volatility,MIXED,Supporting context,"Low-volatility pages had lower decline, but th..."
2,Average search position,MIXED,Context and filtering,"Deep pages had higher decline, but the complet..."
3,CTR relative to position,CONFIRMED,Primary weakness signal,Below-position-band CTR pages showed a meaning...


In [51]:
print("ML-06 FINAL VERDICTS")
print("-" * 60)

for _, row in final_verdict_summary.iterrows():
    print(
        f"{row['signal']}: {row['verdict']} "
        f"| Baseline role: {row['baseline_use']}"
    )

ML-06 FINAL VERDICTS
------------------------------------------------------------
Visibility / impressions: MIXED | Baseline role: Impact weight only
Position volatility: MIXED | Baseline role: Supporting context
Average search position: MIXED | Baseline role: Context and filtering
CTR relative to position: CONFIRMED | Baseline role: Primary weakness signal


## Final Results Summary

The final audit frame contained 76,201 anonymized pages with enough feature-window visibility and data availability.

The distribution analysis showed strong heavy tails. Median feature-window impressions were much lower than the 99th percentile and maximum, so raw volume should not be used without capping, transformation, or percentile ranking.

Visibility received a **MIXED** verdict. The highest-volume bucket had the highest observed decline rate, but the pattern across all buckets was uneven.

Position volatility received a **MIXED** verdict. Low-volatility pages showed the lowest decline rate, but decline did not rise consistently across every higher-volatility bucket.

Average search position received a **MIXED** verdict. Deep-ranking pages showed the highest decline rate, but the other position bands did not follow a consistent order.

CTR relative to position received a **CONFIRMED** verdict. Among sufficiently visible pages ranking between positions 1 and 20, below-position-band CTR was associated with a meaningfully higher observed decline rate.

These results support using position-adjusted CTR as the main weakness signal in the baseline. Visibility should represent potential impact, while volatility and position should provide supporting context.

The findings are observational and intended for decision support. They do not prove search-engine ranking factors or causal refresh impact.

## Leakage and Privacy Check

The declining proxy and outcome-window measurements were used only to evaluate the candidate signals.

They will not be used as inputs in the ML-07 baseline score.

The later baseline may use only feature-window information, including:

- feature impressions;
- feature clicks and CTR;
- average position;
- position volatility;
- position-adjusted CTR gap.

The baseline must exclude:

- outcome impressions;
- outcome daily impressions;
- the declining proxy;
- decline ratios;
- future-window measurements;
- product scores and flags;
- client names, URLs, domains, titles, and raw queries.

In [52]:
allowed_baseline_inputs = {
    "feature_impressions",
    "feature_clicks",
    "feature_ctr",
    "feature_avg_position",
    "feature_position_volatility",
    "feature_available_days",
    "ctr_gap",
    "ctr_status",
}

forbidden_inputs = {
    "outcome_impressions",
    "outcome_daily_impressions",
    "outcome_available_days",
    "is_declining_proxy",
    "decline_ratio",
    "trend_direction",
    "trend_pct",
    "health_score",
    "priority_score",
    "action_type",
    "needs_ctr_fix",
    "refresh_flag",
}

forbidden_overlap = sorted(
    allowed_baseline_inputs.intersection(forbidden_inputs)
)

print("Forbidden inputs found:", forbidden_overlap)

assert len(forbidden_overlap) == 0

print("Leakage check passed.")
print("Only feature-window signals are approved for ML-07.")

Forbidden inputs found: []
Leakage check passed.
Only feature-window signals are approved for ML-07.


In [53]:
print("ML-06 NOTEBOOK RECEIPT")
print("-" * 55)

print("Audit rows:", f"{len(audit_frame):,}")
print(
    "Declining proxy rate:",
    round(audit_frame["is_declining_proxy"].mean(), 3),
)
print("Signal 1 verdict:", signal_1_verdict)
print("Signal 2 verdict:", signal_2_verdict)
print("Signal 3 verdict:", signal_3_verdict)
print("Flag-linked verdict:", flag_verdict)
print("Forbidden inputs found:", forbidden_overlap)

assert len(audit_frame) > 0
assert signal_1_verdict in {
    "CONFIRMED", "OPPOSITE", "MIXED", "FALSE"
}
assert signal_2_verdict in {
    "CONFIRMED", "OPPOSITE", "MIXED", "FALSE"
}
assert signal_3_verdict in {
    "CONFIRMED", "OPPOSITE", "MIXED", "FALSE"
}
assert flag_verdict in {
    "CONFIRMED", "OPPOSITE", "MIXED", "FALSE"
}
assert len(forbidden_overlap) == 0

print("\nML-06 checks passed.")

ML-06 NOTEBOOK RECEIPT
-------------------------------------------------------
Audit rows: 76,201
Declining proxy rate: 0.345
Signal 1 verdict: MIXED
Signal 2 verdict: MIXED
Signal 3 verdict: MIXED
Flag-linked verdict: CONFIRMED
Forbidden inputs found: []

ML-06 checks passed.


# Self-check

- [x] I inspected the distributions before selecting thresholds.
- [x] I identified heavy-tailed signals.
- [x] I tested three safe feature-window signals.
- [x] Every signal table includes `n`.
- [x] Every signal has a one-word verdict.
- [x] I tested CTR relative to position as a real flag-linked signal.
- [x] I restricted the CTR test to sufficiently visible positions 1–20.
- [x] I used the outcome proxy only for evaluation.
- [x] I did not approve any future-window or label-derived baseline input.
- [x] I included no client names, URLs, domains, titles, or private queries.
- [x] My conclusions use observed, measured, directional, and decision-support language.
- [x] The notebook runs from top to bottom without errors.